# Nível 1 do STRATA residual como GGUF Q5_K, numa T4

Antes de rodar: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois *Executar tudo*.

`tests/ppl/strata_to_gguf.py` lê o arquivo residual até o nível 1 (estrato 0 = Q4_K_M) e grava um GGUF comum: as matrizes Q4_K viram Q5_K, as Q6_K ficam como estão. O arquivo sai com a mesma mistura de tipos e o mesmo tamanho de um Q5_K_M, então roda no llama.cpp sem mudança. A tabela mede, no wikitext-2 de teste inteiro: BF16, Q4_K_M, Q5_K_M e o GGUF convertido. Leva uns 30 minutos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q -U transformers datasets safetensors
!pip install -q -e llama.cpp/gguf-py
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /dev/null && cmake --build build -j --target llama-quantize > /dev/null && ls build/bin

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download("unsloth/Llama-3.2-1B", allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
!mkdir -p /content/gguf && python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile /content/gguf/bf16.gguf 2>&1 | tail -1
for q in ["Q4_K_M", "Q5_K_M"]:
    !llama.cpp/build/bin/llama-quantize /content/gguf/bf16.gguf /content/gguf/{q}.gguf {q} > /content/gguf/{q}.log 2>&1
%cd /content/strata
!python tests/ppl/strata_to_gguf.py --model unsloth/Llama-3.2-1B --gguf /content/gguf/Q4_K_M.gguf --out /content/gguf/nivel1_q5k.gguf --no-sizes
!ls -la /content/gguf

In [ ]:
!python tests/ppl/strata_ppl.py --model unsloth/Llama-3.2-1B --depths '' --int-bits '' --gguf Q4_K_M=/content/gguf/Q4_K_M.gguf,Q5_K_M=/content/gguf/Q5_K_M.gguf,STRATA_nivel1_Q5K=/content/gguf/nivel1_q5k.gguf --tag nivel1_q5k --out /content/res
from IPython.display import Markdown, display
display(Markdown(open('/content/res/unsloth__Llama-3.2-1B_nivel1_q5k.md').read()))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/unsloth__Llama-3.2-1B_nivel1_q5k.md')
files.download('/content/res/unsloth__Llama-3.2-1B_nivel1_q5k.json')